# 🛒 Basket Score Model — CATALINA Hackathon

**Challenge:** How can transactional data and external data be used to steer and optimize an omnichannel activation strategy and measure its marketing performance?

**Our Solution:** Evaluate the overall quality of a shopper's basket using NutriScore and EcoScore from Open Food Facts.

## Pipeline Overview
1. Load transactional data + Open Food Facts enriched dataset
2. Encode NutriScore & EcoScore into numeric scales
3. Compute a **Basket Score** per transaction (health + sustainability)
4. Aggregate to shopper-level basket profiles
5. Merge with behavioral segmentation (Premium Loyal / Regular Engaged / Occasional)
6. Define personalized **omnichannel activation strategies** per profile
7. Generate individual shopper recommendations
8. Build a **KPI measurement framework**

In [ ]:
# ─── DEPENDENCIES ───────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyarrow'])

In [ ]:
# ─── IMPORTS ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = {'Premium Loyal': '#2ecc71', 'Regular Engaged': '#3498db', 'Occasional': '#e74c3c'}
SCORE_PALETTE = {'High': '#27ae60', 'Medium': '#f39c12', 'Low': '#c0392b'}
print('✅ Imports ready')

## Step 1 — Load Data

In [ ]:
print('=' * 80)
print('STEP 1: LOADING DATA')
print('=' * 80)

# Core transactional tables
overall_receipt  = pd.read_parquet('ESCP_OVERALL_RECEIPT_2026_V2.parquet')
detailed_receipt = pd.read_parquet('ESCP_DETAILED_RECEIPT_2026_V2.parquet')
events           = pd.read_parquet('ESCP_EVENTS_2026_V2.parquet')

# Normalise transactional_key to plain string (parquet stores as Decimal)
for df in [overall_receipt, detailed_receipt]:
    df['transactional_key'] = df['transactional_key'].astype(str)

# Open Food Facts — pre-joined with transactional data (provided by Catalina)
enriched = pd.read_csv('../ESCP_DETAILED_OPENFOODFACTS.csv', low_memory=False)
enriched['transactional_key'] = enriched['transactional_key'].astype(str)

# Open Food Facts — full reference dataset for extended barcode matching
off_full = pd.read_parquet('../OPEN_FOOD_FACTS_FINAL.parquet')

# Existing behavioral segments (from Segmentation_Model.ipynb)
segments = pd.read_csv('shopper_segments.csv', low_memory=False)

print(f'Overall receipts  : {len(overall_receipt):>10,} rows')
print(f'Detailed receipts : {len(detailed_receipt):>10,} rows')
print(f'Enriched (OFF)    : {len(enriched):>10,} rows  ← pre-joined with food facts')
print(f'OFF full reference: {len(off_full):>10,} rows')
print(f'Events            : {len(events):>10,} rows')
print(f'Shopper segments  : {len(segments):>10,} shoppers')

## Step 2 — Extend Coverage via Full OFF Join

The provided `ESCP_DETAILED_OPENFOODFACTS.csv` is a curated sample. We extend it by joining the full `detailed_receipt` against `OPEN_FOOD_FACTS_FINAL.parquet` on barcode, then union both sources to maximise product coverage.

In [ ]:
print('=' * 80)
print('STEP 2: EXTENDING PRODUCT COVERAGE WITH FULL OFF JOIN')
print('=' * 80)

# Normalise barcode: convert float like 3124480196262.0 → '3124480196262'
# Uses Int64 (nullable) to preserve NaN rows without index drift
def normalise_barcode(series):
    return (
        pd.to_numeric(series, errors='coerce')
          .astype('Int64')         # nullable integer — keeps NaN as pd.NA
          .astype(str)
          .replace({'<NA>': np.nan})
    )

# --- Clean OFF reference ---
off_ref = off_full[['code', 'product_name', 'brands', 'categories',
                     'categories_tags', 'nutriscore_grade', 'environmental_score_grade']].copy()
off_ref['barcode'] = off_ref['code'].astype(str).str.strip()
off_ref = off_ref.drop_duplicates('barcode')

# --- Clean detailed_receipt and join ---
dr = detailed_receipt.copy()
dr['barcode'] = normalise_barcode(dr['upc_with_check_digit'])
dr_valid = dr.dropna(subset=['barcode']).copy()

full_enriched = dr_valid.merge(off_ref, on='barcode', how='left')
match_rate_full = full_enriched['nutriscore_grade'].notna().mean()
print(f'Full join rows     : {len(full_enriched):,}')
print(f'Nutriscore coverage: {match_rate_full:.1%}')

# --- Provided enriched file ---
enriched['barcode'] = normalise_barcode(enriched['upc_with_check_digit'])
enriched_clean = enriched[[
    'transactional_key', 'channel', 'barcode',
    'purch_amt', 'purch_qty', 'price',
    'product_name', 'brands', 'categories', 'categories_tags',
    'nutriscore_grade', 'environmental_score_grade'
]].copy()

full_enriched_clean = full_enriched[[
    'transactional_key', 'channel', 'barcode',
    'purch_amt', 'purch_qty', 'price',
    'product_name', 'brands', 'categories', 'categories_tags',
    'nutriscore_grade', 'environmental_score_grade'
]].copy()

# Union: use provided file first, supplement with full join for remaining transactions
provided_keys = set(enriched_clean['transactional_key'].unique())
supplement = full_enriched_clean[~full_enriched_clean['transactional_key'].isin(provided_keys)]
combined = pd.concat([enriched_clean, supplement], ignore_index=True)

print(f'Combined rows      : {len(combined):,}')
print(f'Transactions covered: {combined["transactional_key"].nunique():,}')
print(f'NutriScore coverage: {combined["nutriscore_grade"].notna().mean():.1%}')
print(f'EcoScore coverage  : {combined["environmental_score_grade"].notna().mean():.1%}')

## Step 3 — Score Encoding

| Grade | NutriScore | EcoScore | Meaning |
|-------|-----------|----------|--------|
| A+ / A | 5 | 5 | Excellent |
| B | 4 | 4 | Good |
| C | 3 | 3 | Average |
| D | 2 | 2 | Poor |
| E / F | 1 | 1 | Very poor |
| unknown / not-applicable | NaN | NaN | Excluded from average |

In [ ]:
print('=' * 80)
print('STEP 3: ENCODING NUTRISCORE & ECOSCORE')
print('=' * 80)

NUTRI_MAP = {'a': 5, 'b': 4, 'c': 3, 'd': 2, 'e': 1}
ECO_MAP   = {'a-plus': 5, 'a': 5, 'b': 4, 'c': 3, 'd': 2, 'e': 1, 'f': 1}

combined['nutri_score_num'] = (
    combined['nutriscore_grade'].str.lower().map(NUTRI_MAP)
)
combined['eco_score_num'] = (
    combined['environmental_score_grade'].str.lower().map(ECO_MAP)
)

# Product-level composite score (60% health, 40% sustainability)
combined['product_score'] = np.where(
    combined['nutri_score_num'].notna() & combined['eco_score_num'].notna(),
    0.6 * combined['nutri_score_num'] + 0.4 * combined['eco_score_num'],
    np.where(
        combined['nutri_score_num'].notna(),
        combined['nutri_score_num'],
        combined['eco_score_num']
    )
)

print('NutriScore distribution (numeric):')
print(combined['nutri_score_num'].value_counts().sort_index())
print()
print('EcoScore distribution (numeric):')
print(combined['eco_score_num'].value_counts().sort_index())
print()
print(f'Product score — mean: {combined["product_score"].mean():.2f}, std: {combined["product_score"].std():.2f}')

## Step 4 — Compute Transaction-Level Basket Score

In [ ]:
print('=' * 80)
print('STEP 4: COMPUTING BASKET SCORES PER TRANSACTION')
print('=' * 80)

# Only keep products with a score
scoreable = combined[combined['product_score'].notna()].copy()

# ── Vectorised spend-weighted basket score ────────────────────────────────────
# weight_i = purch_amt_i / sum(purch_amt) per transaction
scoreable['purch_amt_clipped'] = scoreable['purch_amt'].clip(lower=0)

# Per-transaction total spend (for weighting)
tx_spend = scoreable.groupby('transactional_key')['purch_amt_clipped'].transform('sum')
scoreable['weight'] = np.where(tx_spend > 0, scoreable['purch_amt_clipped'] / tx_spend, 1.0)
scoreable['weighted_score'] = scoreable['product_score'] * scoreable['weight']

# Aggregate per transaction
basket_scores = (
    scoreable.groupby('transactional_key', sort=False)
    .agg(
        basket_score        = ('weighted_score',    'sum'),
        basket_nutri_score  = ('nutri_score_num',   'mean'),
        basket_eco_score    = ('eco_score_num',     'mean'),
        n_products_total    = ('product_score',     'count'),
        pct_products_scored = ('product_score',     lambda x: x.notna().mean()),
        basket_spend        = ('purch_amt',         'sum'),
    )
    .reset_index()
)
basket_scores['n_products_scored'] = basket_scores['n_products_total']  # all rows here have a score

print(f'Transactions scored: {len(basket_scores):,}')
print(f'Basket score — mean  : {basket_scores["basket_score"].mean():.2f}')
print(f'Basket score — median: {basket_scores["basket_score"].median():.2f}')
print(f'Basket score — std   : {basket_scores["basket_score"].std():.2f}')

# Attach shopper ID and date from overall_receipt
overall_slim = overall_receipt[['transactional_key', 'loyalty_card_key', 'date', 'channel', 'store_key']].copy()
overall_slim['date'] = pd.to_datetime(overall_slim['date'])

basket_scores = basket_scores.merge(overall_slim, on='transactional_key', how='left')
basket_scores = basket_scores.rename(columns={'loyalty_card_key': 'shopper_id'})

print(f'Shoppers with ≥1 scored basket: {basket_scores["shopper_id"].nunique():,}')

## Step 5 — Shopper-Level Basket Profile

In [ ]:
print('=' * 80)
print('STEP 5: AGGREGATING TO SHOPPER-LEVEL BASKET PROFILE')
print('=' * 80)

ref_date = basket_scores['date'].max()
cutoff_recent = ref_date - pd.Timedelta(days=30)

# All-time profile
shopper_basket = (
    basket_scores
    .groupby('shopper_id')
    .agg(
        avg_basket_score       = ('basket_score', 'mean'),
        median_basket_score    = ('basket_score', 'median'),
        std_basket_score       = ('basket_score', 'std'),
        avg_nutri_score        = ('basket_nutri_score', 'mean'),
        avg_eco_score          = ('basket_eco_score', 'mean'),
        n_scored_baskets       = ('basket_score', 'count'),
        avg_pct_scored         = ('pct_products_scored', 'mean'),
        total_scored_spend     = ('basket_spend', 'sum'),
    )
    .reset_index()
)

# Recent trend (last 30 days)
recent = (
    basket_scores[basket_scores['date'] >= cutoff_recent]
    .groupby('shopper_id')['basket_score']
    .mean()
    .reset_index()
    .rename(columns={'basket_score': 'recent_avg_basket_score'})
)

# Early-period score (everything before last 30 days)
early = (
    basket_scores[basket_scores['date'] < cutoff_recent]
    .groupby('shopper_id')['basket_score']
    .mean()
    .reset_index()
    .rename(columns={'basket_score': 'early_avg_basket_score'})
)

shopper_basket = shopper_basket.merge(recent, on='shopper_id', how='left')
shopper_basket = shopper_basket.merge(early, on='shopper_id', how='left')

# Score trend: positive = improving, negative = declining
shopper_basket['basket_score_trend'] = (
    shopper_basket['recent_avg_basket_score'] - shopper_basket['early_avg_basket_score']
)

# Score tier (on 1-5 scale)
def score_tier(score):
    if pd.isna(score): return 'Unknown'
    if score >= 3.5:   return 'High'
    if score >= 2.5:   return 'Medium'
    return 'Low'

shopper_basket['basket_score_tier'] = shopper_basket['avg_basket_score'].apply(score_tier)

print(f'Shoppers profiled: {len(shopper_basket):,}')
print()
print('Basket score tier distribution:')
print(shopper_basket['basket_score_tier'].value_counts())
print()
print(shopper_basket[['avg_basket_score', 'avg_nutri_score', 'avg_eco_score',
                       'basket_score_trend', 'n_scored_baskets']].describe().round(2))

## Step 6 — Merge with Behavioral Segments

In [ ]:
print('=' * 80)
print('STEP 6: BUILDING UNIFIED SHOPPER PROFILE')
print('=' * 80)

# segments may use 'shopper_id' OR 'loyalty_card_key' as key
seg_key = 'shopper_id' if 'shopper_id' in segments.columns else 'loyalty_card_key'
segments_slim = segments.rename(columns={seg_key: 'shopper_id'})[
    ['shopper_id', 'segment', 'total_spend', 'avg_transaction_value',
     'num_transactions', 'frequency_per_month', 'pct_ecommerce',
     'primary_store_loyalty', 'offers_activated', 'offers_redeemed', 'redemption_rate']
].copy()

# Convert shopper_id to string for consistent join
segments_slim['shopper_id'] = segments_slim['shopper_id'].astype(str)
shopper_basket['shopper_id']   = shopper_basket['shopper_id'].astype(str)

# Full outer join: keep all shoppers even if they lack basket scores
full_profile = segments_slim.merge(shopper_basket, on='shopper_id', how='left')
full_profile['basket_score_tier'] = full_profile['basket_score_tier'].fillna('Unknown')

# Composite profile label: Segment × Basket Tier
full_profile['profile'] = full_profile['segment'] + ' | ' + full_profile['basket_score_tier']

print(f'Full profile rows: {len(full_profile):,}')
print(f'Shoppers with basket score: {full_profile["avg_basket_score"].notna().sum():,} '
      f'({full_profile["avg_basket_score"].notna().mean():.1%})')
print()
print('Segment × Basket Score Tier matrix:')
matrix = pd.crosstab(full_profile['segment'], full_profile['basket_score_tier'],
                     margins=True, margins_name='Total')
print(matrix)

## Step 7 — Omnichannel Activation Strategy Engine

For each **Segment × Basket Score Tier** combination, we define:
- **Channel priority** (In-store / E-commerce / Push notification)
- **Timing** (purchase day, frequency)
- **Incentive type** (discount, earn points, mission)
- **Message theme** (health upgrade, sustainability, loyalty reward)
- **Shopping mission** (specific product swap or addition)

In [ ]:
print('=' * 80)
print('STEP 7: DEFINING ACTIVATION STRATEGIES')
print('=' * 80)

ACTIVATION_PLAYBOOK = {
    # ── PREMIUM LOYAL ────────────────────────────────────────────────────────
    ('Premium Loyal', 'High'): {
        'channel':       'In-store + Loyalty App',
        'timing':        'Day before regular shopping day',
        'incentive':     'Bonus loyalty points on next scored basket',
        'message_theme': 'Sustainability Ambassador — reward & celebrate',
        'mission':       'Maintain score ≥ 3.5 — earn Premium Green badge',
        'kpi_target':    'Retain score, increase basket spend by 5%',
    },
    ('Premium Loyal', 'Medium'): {
        'channel':       'In-store digital shelf + App push',
        'timing':        'At checkout (triggered by basket composition)',
        'incentive':     '10% off on healthier alternative product',
        'message_theme': 'One small swap, big impact',
        'mission':       'Replace 1 D/E NutriScore product with B/C equivalent',
        'kpi_target':    'Lift basket score by +0.3 within 4 weeks',
    },
    ('Premium Loyal', 'Low'): {
        'channel':       'App + Email (pre-shop planning)',
        'timing':        '2 days before typical shopping day',
        'incentive':     'Personalised meal plan + 15% off on A/B products',
        'message_theme': 'Your health upgrade journey starts here',
        'mission':       'Add 2 NutriScore A/B products to next basket',
        'kpi_target':    'Lift basket score by +0.5 within 8 weeks',
    },
    # ── REGULAR ENGAGED ──────────────────────────────────────────────────────
    ('Regular Engaged', 'High'): {
        'channel':       'App + E-commerce homepage banner',
        'timing':        'When browsing online or entering store',
        'incentive':     'Free delivery or express pick-up for green baskets',
        'message_theme': 'Shop smarter — unlock green basket perks',
        'mission':       'Complete 3 green baskets to unlock Eco Shopper status',
        'kpi_target':    'Increase e-commerce channel share by 10%',
    },
    ('Regular Engaged', 'Medium'): {
        'channel':       'In-store aisle nudge (digital) + App',
        'timing':        'During shopping trip',
        'incentive':     'Swap & Save: €0.50 off when swapping to healthier item',
        'message_theme': 'Better choices, same price',
        'mission':       'Improve at least one category score (e.g., dairy, snacks)',
        'kpi_target':    'Lift basket score by +0.3 in 6 weeks',
    },
    ('Regular Engaged', 'Low'): {
        'channel':       'Email + App onboarding flow',
        'timing':        'Post-purchase email within 24h',
        'incentive':     'Starter pack: 3x €1 vouchers for healthier categories',
        'message_theme': 'Small changes, lasting habits',
        'mission':       'Complete "Healthy Week" mission — 5 scoring products',
        'kpi_target':    'Lift basket score by +0.5 in 8 weeks; retain shopper',
    },
    # ── OCCASIONAL ────────────────────────────────────────────────────────────
    ('Occasional', 'High'): {
        'channel':       'Email re-engagement',
        'timing':        '2 weeks after last purchase',
        'incentive':     '€5 off next visit (reward quality shopper)',
        'message_theme': 'We miss you — come back for more green choices',
        'mission':       'Reactivation: next scored basket earns double points',
        'kpi_target':    'Increase visit frequency by 20%',
    },
    ('Occasional', 'Medium'): {
        'channel':       'SMS / Push on shopping day',
        'timing':        'Morning of likely shopping day (model-predicted)',
        'incentive':     '€2 off when basket score ≥ 3.0',
        'message_theme': 'Today is a great day for a healthy shop',
        'mission':       'First health mission: 3 products with A or B score',
        'kpi_target':    'Increase visit frequency + raise basket score',
    },
    ('Occasional', 'Low'): {
        'channel':       'In-store POS + Loyalty sign-up prompt',
        'timing':        'At checkout on next visit',
        'incentive':     'Sign-up offer: instant €3 off + health score tracker',
        'message_theme': 'Start your wellness journey — every swap counts',
        'mission':       'Replace 1 ultra-processed product with natural alternative',
        'kpi_target':    'Convert to Regular Engaged in 3 months',
    },
    # ── FALLBACK (Unknown score) ──────────────────────────────────────────────
    ('Premium Loyal', 'Unknown'): {
        'channel':       'App + In-store',
        'timing':        'Next visit',
        'incentive':     'Scan & Earn: scan products to unlock your health score',
        'message_theme': 'Discover your basket health score',
        'mission':       'Scan 5 products with the app to unlock personalised offers',
        'kpi_target':    'Achieve product score coverage > 50%',
    },
}

# Default for any unmapped combination
DEFAULT_STRATEGY = {
    'channel':       'App + In-store',
    'timing':        'Next visit',
    'incentive':     'Scan & Earn: scan products to unlock your health score',
    'message_theme': 'Discover your basket health score',
    'mission':       'Scan 5 products with the app to unlock personalised offers',
    'kpi_target':    'Achieve product score coverage > 50%',
}

def lookup_strategy(segment, tier):
    return ACTIVATION_PLAYBOOK.get((segment, tier), DEFAULT_STRATEGY)

print('Activation playbook defined:')
for (seg, tier), strat in ACTIVATION_PLAYBOOK.items():
    print(f'  [{seg} | {tier}]  → {strat["incentive"]}')

## Step 8 — Generate Per-Shopper Recommendations

In [ ]:
print('=' * 80)
print('STEP 8: GENERATING PERSONALISED ACTIVATION RECOMMENDATIONS')
print('=' * 80)

strategy_fields = ['channel', 'timing', 'incentive', 'message_theme', 'mission', 'kpi_target']

for field in strategy_fields:
    full_profile[f'strat_{field}'] = full_profile.apply(
        lambda row: lookup_strategy(row['segment'], row['basket_score_tier'])[field],
        axis=1
    )

# Preferred channel override: if shopper is heavy e-commerce user, shift to digital
full_profile['strat_channel'] = np.where(
    full_profile['pct_ecommerce'] > 0.3,
    full_profile['strat_channel'].str.replace('In-store', 'E-commerce', regex=False),
    full_profile['strat_channel']
)

# Trend-based message boost
full_profile['trend_note'] = np.where(
    full_profile['basket_score_trend'] > 0.2,
    ' 📈 Score improving — reinforce positive momentum',
    np.where(
        full_profile['basket_score_trend'] < -0.2,
        ' 📉 Score declining — re-engage with targeted nudge',
        ''
    )
)

print(f'Recommendations generated for {len(full_profile):,} shoppers')
print()
print('Sample recommendations:')
sample_cols = ['shopper_id', 'segment', 'avg_basket_score', 'basket_score_tier',
               'strat_channel', 'strat_incentive', 'strat_mission']
print(full_profile[sample_cols].dropna(subset=['avg_basket_score']).head(10).to_string(index=False))

## Step 9 — Visualisations

In [ ]:
# ── VIZ 1: Basket Score Distribution by Segment ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Distribution
ax = axes[0]
for seg, color in PALETTE.items():
    data = full_profile[(full_profile['segment'] == seg) & full_profile['avg_basket_score'].notna()]
    ax.hist(data['avg_basket_score'], bins=30, alpha=0.6, label=seg, color=color, edgecolor='white')
ax.set_xlabel('Average Basket Score (1–5)', fontsize=12)
ax.set_ylabel('Number of Shoppers', fontsize=12)
ax.set_title('Basket Score Distribution by Segment', fontsize=13, fontweight='bold')
ax.legend()
ax.axvline(2.5, color='red', linestyle='--', alpha=0.5, label='Low threshold')
ax.axvline(3.5, color='green', linestyle='--', alpha=0.5, label='High threshold')

# Right: Box plot
ax2 = axes[1]
segments_order = ['Premium Loyal', 'Regular Engaged', 'Occasional']
plot_data = [full_profile[(full_profile['segment'] == s) & full_profile['avg_basket_score'].notna()]['avg_basket_score'].values
             for s in segments_order]
bp = ax2.boxplot(plot_data, tick_labels=segments_order, patch_artist=True, widths=0.5)
for patch, seg in zip(bp['boxes'], segments_order):
    patch.set_facecolor(PALETTE[seg])
    patch.set_alpha(0.8)
ax2.set_ylabel('Average Basket Score (1–5)', fontsize=12)
ax2.set_title('Basket Score by Behavioural Segment', fontsize=13, fontweight='bold')
ax2.axhline(3.5, color='green', linestyle='--', alpha=0.4)
ax2.axhline(2.5, color='red', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('06_basket_score_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 06_basket_score_by_segment.png')

In [ ]:
# ── VIZ 2: Segment × Basket Tier Heatmap ─────────────────────────────────────
scored = full_profile[full_profile['basket_score_tier'] != 'Unknown']
heatmap_data = pd.crosstab(scored['segment'], scored['basket_score_tier'])
# Reorder columns
for col in ['High', 'Medium', 'Low']:
    if col not in heatmap_data.columns:
        heatmap_data[col] = 0
heatmap_data = heatmap_data[['High', 'Medium', 'Low']]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heatmap_data, annot=True, fmt=',d', cmap='YlGn', ax=ax,
            linewidths=1, linecolor='white', cbar_kws={'label': 'Number of Shoppers'})
ax.set_title('Shopper Count: Behavioural Segment × Basket Score Tier', fontsize=13, fontweight='bold')
ax.set_xlabel('Basket Score Tier', fontsize=11)
ax.set_ylabel('Behavioural Segment', fontsize=11)
plt.tight_layout()
plt.savefig('07_segment_score_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_segment_score_heatmap.png')

In [ ]:
# ── VIZ 3: NutriScore vs EcoScore scatter ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
scored_full = full_profile[
    full_profile['avg_nutri_score'].notna() & full_profile['avg_eco_score'].notna()
].sample(min(5000, len(full_profile)), random_state=42)

for seg, color in PALETTE.items():
    d = scored_full[scored_full['segment'] == seg]
    ax.scatter(d['avg_nutri_score'], d['avg_eco_score'],
               alpha=0.4, s=20, color=color, label=seg)

ax.set_xlabel('Average NutriScore (1=E, 5=A)', fontsize=12)
ax.set_ylabel('Average EcoScore (1=E/F, 5=A+)', fontsize=12)
ax.set_title('Shopper Health vs Sustainability Profile\n(sample of 5,000 shoppers)', fontsize=13, fontweight='bold')
ax.axvline(3, color='orange', linestyle='--', alpha=0.4)
ax.axhline(3, color='orange', linestyle='--', alpha=0.4)

# Quadrant labels
for x, y, label in [(1.5, 4.5, 'Eco-friendly\nUnhealthy'), (4.5, 4.5, 'Eco-friendly\n& Healthy'),
                     (1.5, 1.5, 'Neither'), (4.5, 1.5, 'Healthy\nNot Eco')]:
    ax.text(x, y, label, ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('08_nutri_vs_eco.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 08_nutri_vs_eco.png')

In [ ]:
# ── VIZ 4: Basket Score vs Total Spend ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
sample = full_profile[full_profile['avg_basket_score'].notna()].sample(min(8000, len(full_profile)), random_state=42)

for seg, color in PALETTE.items():
    d = sample[sample['segment'] == seg]
    ax.scatter(d['avg_basket_score'], d['total_spend'],
               alpha=0.4, s=25, color=color, label=seg)

ax.set_xlabel('Average Basket Score (1–5)', fontsize=12)
ax.set_ylabel('Total Spend (€)', fontsize=12)
ax.set_title('Basket Quality vs Shopper Spend Value', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, full_profile['total_spend'].quantile(0.99))
plt.tight_layout()
plt.savefig('09_score_vs_spend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 09_score_vs_spend.png')

In [ ]:
# ── VIZ 5: Channel Breakdown by Profile ──────────────────────────────────────
channel_alloc = full_profile.groupby(['segment', 'basket_score_tier'])['strat_channel'].first().reset_index()
channel_alloc = channel_alloc[channel_alloc['basket_score_tier'] != 'Unknown']

fig, ax = plt.subplots(figsize=(14, 6))
profiles = channel_alloc['segment'] + '\n' + channel_alloc['basket_score_tier']
channels = channel_alloc['strat_channel']

unique_channels = channels.unique()
cmap = plt.cm.Set2
ch_colors = {ch: cmap(i / len(unique_channels)) for i, ch in enumerate(unique_channels)}

bars = ax.barh(profiles, [1]*len(profiles), color=[ch_colors[c] for c in channels], edgecolor='white', height=0.6)
for i, (bar, ch) in enumerate(zip(bars, channels)):
    ax.text(0.5, i, ch, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.set_xlabel('')
ax.set_xticks([])
ax.set_title('Recommended Activation Channel by Shopper Profile', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('10_activation_channels.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 10_activation_channels.png')

## Step 10 — KPI Measurement Framework

To measure the effectiveness of the omnichannel activation strategy, we define **before/after KPIs** at the shopper and campaign level.

In [ ]:
print('=' * 80)
print('STEP 10: KPI MEASUREMENT FRAMEWORK')
print('=' * 80)

# ── Baseline KPIs per profile ────────────────────────────────────────────────
kpi_baseline = (
    full_profile[full_profile['basket_score_tier'] != 'Unknown']
    .groupby(['segment', 'basket_score_tier'])
    .agg(
        n_shoppers           = ('shopper_id', 'count'),
        avg_basket_score     = ('avg_basket_score', 'mean'),
        avg_total_spend      = ('total_spend', 'mean'),
        avg_frequency        = ('frequency_per_month', 'mean'),
        avg_redemption_rate  = ('redemption_rate', 'mean'),
        pct_ecommerce        = ('pct_ecommerce', 'mean'),
        avg_nutri_score      = ('avg_nutri_score', 'mean'),
        avg_eco_score        = ('avg_eco_score', 'mean'),
    )
    .reset_index()
    .round(3)
)

# Add strategy targets
kpi_baseline['activation_target'] = kpi_baseline.apply(
    lambda r: lookup_strategy(r['segment'], r['basket_score_tier'])['kpi_target'], axis=1
)

print('BASELINE KPIs BY PROFILE:')
print(kpi_baseline.to_string(index=False))

print()
print('─' * 80)
print('MEASUREMENT PLAN:')
print('─' * 80)
print("""
PRIMARY METRICS (measured T+4 weeks, T+8 weeks, T+12 weeks):
  1. Δ avg_basket_score          — core outcome (health + sustainability improvement)
  2. Δ total_spend               — revenue impact
  3. Δ frequency_per_month       — engagement / retention
  4. offer_redemption_rate       — campaign effectiveness
  5. segment_upgrade_rate        — % Occasional → Regular Engaged

SECONDARY METRICS:
  6. channel_shift_rate          — % migrating to preferred channel
  7. category_score_improvement  — which categories improved most
  8. mission_completion_rate     — % completing assigned shopping mission
  9. nps_delta                   — net promoter score change

CONTROL GROUP:
  - 10% holdout (no activation) per profile for A/B test validity
  - Randomised at shopper_id level within each Segment × Tier cell

REPORTING CADENCE:
  - Weekly: redemption rate, basket score movement
  - Monthly: spend, frequency, segment migration
  - Quarterly: LTV impact, sustainability impact report
""")

In [ ]:
# ── VIZ 6: KPI Dashboard ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Baseline KPI Dashboard by Shopper Profile', fontsize=15, fontweight='bold')

tier_order = ['High', 'Medium', 'Low']
seg_order = ['Premium Loyal', 'Regular Engaged', 'Occasional']

metrics = [
    ('n_shoppers',          'Number of Shoppers',         'Blues'),
    ('avg_basket_score',    'Avg Basket Score (1–5)',      'Greens'),
    ('avg_total_spend',     'Avg Total Spend (€)',         'Oranges'),
    ('avg_frequency',       'Avg Purchase Frequency/mo',  'Purples'),
    ('avg_redemption_rate', 'Avg Offer Redemption Rate',  'Reds'),
    ('pct_ecommerce',       '% E-commerce Transactions',  'YlOrBr'),
]

for ax, (metric, title, cmap) in zip(axes.flat, metrics):
    pivot = kpi_baseline.pivot(index='segment', columns='basket_score_tier', values=metric)
    for col in tier_order:
        if col not in pivot.columns:
            pivot[col] = np.nan
    pivot = pivot.reindex(columns=tier_order)
    pivot = pivot.reindex(seg_order)
    sns.heatmap(pivot, annot=True, fmt='.2f' if metric != 'n_shoppers' else ',.0f',
                cmap=cmap, ax=ax, linewidths=0.5, linecolor='white')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Basket Score Tier')
    ax.set_ylabel('Segment')

plt.tight_layout()
plt.savefig('11_kpi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 11_kpi_dashboard.png')

## Step 11 — Category-Level Deep Dive

Identify which product categories drag the basket score down — enabling targeted swap suggestions.

In [ ]:
print('=' * 80)
print('STEP 11: CATEGORY-LEVEL SCORE ANALYSIS')
print('=' * 80)

# Extract top-level category from categories_tags
def extract_top_category(tags):
    if pd.isna(tags):
        return 'Unknown'
    parts = str(tags).split(',')
    for part in parts:
        clean = part.strip().replace('en:', '').replace('-', ' ').title()
        if len(clean) > 3 and 'Foods' not in clean:
            return clean
    return 'Other'

combined['top_category'] = combined['categories_tags'].apply(extract_top_category)

cat_scores = (
    combined[combined['product_score'].notna()]
    .groupby('top_category')
    .agg(
        avg_product_score = ('product_score', 'mean'),
        avg_nutri         = ('nutri_score_num', 'mean'),
        avg_eco           = ('eco_score_num', 'mean'),
        n_products        = ('product_score', 'count'),
        total_spend       = ('purch_amt', 'sum'),
    )
    .reset_index()
    .sort_values('total_spend', ascending=False)
)

# Top 20 categories by spend
top_cats = cat_scores[cat_scores['n_products'] >= 50].head(20)

print('Top 20 categories by spend (with score ≥ 50 products):')
print(top_cats[['top_category', 'avg_product_score', 'avg_nutri', 'avg_eco', 'n_products', 'total_spend']]
      .to_string(index=False))

# ── Viz: Category scores (top 20 by spend) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
colors_bar = [SCORE_PALETTE['High'] if s >= 3.5 else
              SCORE_PALETTE['Medium'] if s >= 2.5 else
              SCORE_PALETTE['Low'] for s in top_cats['avg_product_score']]

bars = ax.barh(top_cats['top_category'], top_cats['avg_product_score'],
               color=colors_bar, edgecolor='white', height=0.7)
ax.axvline(2.5, color='orange', linestyle='--', alpha=0.6, label='Medium threshold')
ax.axvline(3.5, color='green', linestyle='--', alpha=0.6, label='High threshold')
ax.set_xlabel('Average Product Score (1–5)', fontsize=11)
ax.set_title('Top 20 Categories by Spend — Average Health & Sustainability Score', fontsize=12, fontweight='bold')
ax.legend()
for bar, score in zip(bars, top_cats['avg_product_score']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{score:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('12_category_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 12_category_scores.png')

## Step 12 — Save Final Outputs

In [ ]:
print('=' * 80)
print('STEP 12: SAVING OUTPUTS')
print('=' * 80)

# 1. Full shopper profile with activation recommendations
output_cols = [
    'shopper_id', 'segment', 'total_spend', 'avg_transaction_value',
    'num_transactions', 'frequency_per_month', 'pct_ecommerce',
    'redemption_rate', 'avg_basket_score', 'median_basket_score',
    'avg_nutri_score', 'avg_eco_score', 'basket_score_tier',
    'basket_score_trend', 'n_scored_baskets', 'profile',
    'strat_channel', 'strat_timing', 'strat_incentive',
    'strat_message_theme', 'strat_mission', 'strat_kpi_target', 'trend_note',
]
output_cols_available = [c for c in output_cols if c in full_profile.columns]
shopper_activation = full_profile[output_cols_available].copy()
shopper_activation.to_csv('shopper_activation_strategy.csv', index=False)
print(f'✅ Saved: shopper_activation_strategy.csv ({len(shopper_activation):,} rows)')

# 2. KPI baseline
kpi_baseline.to_csv('kpi_baseline.csv', index=False)
print(f'✅ Saved: kpi_baseline.csv')

# 3. Transaction basket scores
basket_scores.to_csv('transaction_basket_scores.csv', index=False)
print(f'✅ Saved: transaction_basket_scores.csv ({len(basket_scores):,} rows)')

# 4. Category analysis
cat_scores.to_csv('category_scores.csv', index=False)
print(f'✅ Saved: category_scores.csv')

print()
print('=' * 80)
print('✅ BASKET SCORE MODEL COMPLETE!')
print('=' * 80)
print()
print('SUMMARY:')
print(f'  Shoppers profiled         : {len(full_profile):,}')
print(f'  Shoppers with basket score: {full_profile["avg_basket_score"].notna().sum():,}')
print(f'  Unique profiles (Seg×Tier): {full_profile["profile"].nunique()}')
print(f'  Transactions scored       : {len(basket_scores):,}')
print(f'  Categories analysed       : {len(cat_scores):,}')

---

## Summary & Business Impact

### What We Built

| Component | Description |
|-----------|-------------|
| **Basket Score** | Per-transaction weighted score combining NutriScore (60%) + EcoScore (40%) on a 1–5 scale |
| **Shopper Profiling** | All-time avg score, recent trend, tier classification (High / Medium / Low) |
| **9-Cell Strategy Matrix** | 3 behavioural segments × 3 score tiers = 9 distinct activation playbooks |
| **Personalised Missions** | Shopping missions (product swaps, category upgrades) per profile |
| **Omnichannel Routing** | Channel selected based on segment + historical e-commerce behaviour |
| **KPI Framework** | Baseline metrics + measurement plan with control group for A/B testing |
| **Category Intelligence** | Top dragging categories identified to fuel swap recommendations |

### Why This Answers the Challenge

- **Transactional data** → who shops when, how often, how much  
- **Open Food Facts** → what they buy and at what quality  
- **Basket Score** → a single actionable metric Catalina can optimise  
- **Activation strategy** → right message, right channel, right moment  
- **KPI framework** → measurable marketing performance over time  